# Downstream evaluation on MoleculeNetFine-tunes each pretrained encoder on the benchmark CSVs and reports severalmetrics per task type, not only ROC-AUC and RMSE.| Task | Metrics || --- | --- || Classification | ROC-AUC, PR-AUC, F1, balanced accuracy, MCC || Regression | RMSE, MAE, R-squared, Pearson r, Spearman rho |PR-AUC matters here because several of these benchmarks are heavilyimbalanced: HIV is about 3.5% actives and ClinTox about 7% toxic. ROC-AUCflatters a classifier on imbalanced data, while PR-AUC does not. MCC isreported for the same reason, being a single number that accounts for all fourcells of the confusion matrix.R-squared is reported alongside RMSE because RMSE alone is not comparableacross datasets with different target variances. An R-squared near zero meansthe model is no better than predicting the mean, which an RMSE figure can hide.## DatasetsColumn names are taken from the uploaded CSVs, which differ from the DeepChemdistribution in one place: **BACE uses `smiles` here, not `mol`**. Task countsand row counts were checked against the APK-MRL tables.Human and C.elegans are excluded: they are drug-target interaction datasetsrequiring a protein encoder, not molecular property prediction.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os

os.environ["PYTHONWARNINGS"] = "ignore"

import json
import subprocess
import sys
import time
from pathlib import Path

for module, package in [("torch_geometric", "torch-geometric"), ("rdkit", "rdkit"),
                        ("sklearn", "scikit-learn"), ("scipy", "scipy")]:
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")

WORKSPACE = Path("/workspace") if Path("/workspace").exists() else Path.cwd()
DATA_DIR = WORKSPACE / "data" / "MoleculeNet"
CACHE_DIR = WORKSPACE / "data" / "downstream_cache"
RESULTS_DIR = WORKSPACE / "runs" / "downstream"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | data: {DATA_DIR}")

## Benchmark registry

In [ ]:
# filename, smiles column, target columns (None = all remaining), task type,
# split strategy, and the metric the literature reports for this dataset.
DATASETS = {
    # ---- classification, scaffold split, ROC-AUC ----
    "bace":      dict(file="bace/bace.csv",                 smiles="smiles",
                      targets=["Class"],                    task="classification",
                      split="scaffold", primary="roc_auc"),
    "bbbp":      dict(file="bbbp/BBBP.csv",                 smiles="smiles",
                      targets=["p_np"],                     task="classification",
                      split="scaffold", primary="roc_auc"),
    "clintox":   dict(file="clintox/clintox.csv",           smiles="smiles",
                      targets=["FDA_APPROVED", "CT_TOX"],   task="classification",
                      split="scaffold", primary="roc_auc"),
    "hiv":       dict(file="hiv/HIV.csv",                   smiles="smiles",
                      targets=["HIV_active"],               task="classification",
                      split="scaffold", primary="roc_auc"),
    "tox21":     dict(file="tox21/tox21.csv",               smiles="smiles",
                      targets=None, drop=["mol_id"],        task="classification",
                      split="scaffold", primary="roc_auc"),
    "sider":     dict(file="sider/sider.csv",               smiles="smiles",
                      targets=None, drop=[],                task="classification",
                      split="scaffold", primary="roc_auc"),

    # ---- regression ----
    "esol":      dict(file="esol/esol.csv",                 smiles="smiles",
                      targets=["measured log solubility in mols per litre"],
                      task="regression", split="scaffold",  primary="rmse"),
    "freesolv":  dict(file="freesolv/freesolv.csv",         smiles="smiles",
                      targets=["expt"],                     task="regression",
                      split="scaffold", primary="rmse"),
    "lipophilicity": dict(file="lipophilicity/Lipophilicity.csv", smiles="smiles",
                      targets=["exp"],                      task="regression",
                      split="scaffold", primary="rmse"),
    # QM7 and QM8 are reported with MAE in the literature, and use a random
    # split rather than a scaffold split.
    "qm7":       dict(file="qm7/qm7.csv",                   smiles="smiles",
                      targets=["u0_atom"],                  task="regression",
                      split="random",   primary="mae"),
    "qm8":       dict(file="qm8/qm8.csv",                   smiles="smiles",
                      targets=None, drop=[],                task="regression",
                      split="random",   primary="mae"),
}

# Pretrained encoders to evaluate. "none" adds a randomly initialised control,
# which is the reference point for whether pretraining helped at all.
CHECKPOINTS = {
    "ManualAug-CL": WORKSPACE / "runs" / "manualaug_cl_fixed" / "best_model.pt",
    "GMACORE": WORKSPACE / "runs" / "gmacore_fixed" / "best_model.pt",
    "GMACORE-TM": WORKSPACE / "runs" / "gmacore_tm_fixedgan" / "best_model.pt",
    "No pretraining": None,
}

ACTIVE_DATASETS = list(DATASETS)      # trim this list for a quick run
SEEDS = [0, 1, 2]
PROTOCOL = "finetune"                 # "finetune" or "linear_probe"
EPOCHS = 100
PATIENCE = 30
BATCH_SIZE = 32
ENCODER_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-6
HEAD_DROPOUT = 0.2

for name in ACTIVE_DATASETS:
    path = DATA_DIR / DATASETS[name]["file"]
    print(f"  {name:<14} {'found' if path.exists() else 'MISSING'}  {path}")

## FeaturizationIdentical to pretraining. The feature statistics saved by each pretraining runare reused, so a molecule is standardised exactly as it was during pretraining.Using downstream statistics instead would shift the input distribution theencoder was trained on.

In [ ]:
NUM_EDGE_FEATURES = 5
ATOM_LIST = list(range(1, 119))
CHIRALITY_LIST = [Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
                  Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
                  Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
                  Chem.rdchem.ChiralType.CHI_OTHER]
BOND_LIST = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
BOND_DIR_LIST = [Chem.rdchem.BondDir.NONE, Chem.rdchem.BondDir.ENDUPRIGHT,
                 Chem.rdchem.BondDir.ENDDOWNRIGHT]

_ATOM_INDEX = {z: i for i, z in enumerate(ATOM_LIST)}
_CHIRAL_INDEX = {c: i for i, c in enumerate(CHIRALITY_LIST)}
_BOND_INDEX = {b: i for i, b in enumerate(BOND_LIST)}
_BOND_DIR_INDEX = {d: i for i, d in enumerate(BOND_DIR_LIST)}
_ELEMENT_CACHE = {}


def element_contributions(symbol):
    if symbol in _ELEMENT_CACHE:
        return _ELEMENT_CACHE[symbol]
    weight, logp = 0.0, 0.0
    probe = Chem.MolFromSmiles(f"[{symbol}]")
    if probe is not None:
        try:
            weight = float(Descriptors.ExactMolWt(probe))
            logp = float(Descriptors.MolLogP(probe))
        except Exception:
            pass
    _ELEMENT_CACHE[symbol] = (weight, logp)
    return weight, logp


def featurize(smiles, label=None, keep_largest_fragment=True):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if keep_largest_fragment and "." in smiles:
        fragments = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
        if len(fragments) > 1:
            mol = max(fragments, key=lambda m: m.GetNumHeavyAtoms())
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        return None
    if mol.GetNumHeavyAtoms() < 2:
        return None

    categorical, physicochemical = [], []
    for atom in mol.GetAtoms():
        categorical.append([_ATOM_INDEX.get(atom.GetAtomicNum(), 0),
                            _CHIRAL_INDEX.get(atom.GetChiralTag(), 0)])
        weight, logp = element_contributions(atom.GetSymbol())
        physicochemical.append([weight, logp, float(atom.GetFormalCharge()),
                                float(int(atom.GetHybridization())),
                                float(atom.GetIsAromatic()), float(atom.GetTotalNumHs()),
                                float(atom.GetTotalValence()), float(atom.GetDegree())])

    rows, cols, features = [], [], []
    for bond in mol.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        rotatable = (bond.GetBondType() == Chem.rdchem.BondType.SINGLE
                     and not bond.IsInRing()
                     and bond.GetBeginAtom().GetDegree() > 1
                     and bond.GetEndAtom().GetDegree() > 1)
        feature = [float(_BOND_INDEX.get(bond.GetBondType(), 0)),
                   float(_BOND_DIR_INDEX.get(bond.GetBondDir(), 0)),
                   float(bond.GetIsConjugated()), float(rotatable), 0.0]
        rows += [start, end]; cols += [end, start]; features += [feature, feature]

    if not rows:
        return None

    data = Data(x_cat=torch.tensor(categorical, dtype=torch.long),
                x_phys=torch.tensor(physicochemical, dtype=torch.float),
                edge_index=torch.tensor([rows, cols], dtype=torch.long),
                edge_attr=torch.tensor(features, dtype=torch.float),
                num_nodes=len(categorical))
    data.smiles = smiles
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data


print("Featurizer ready.")

In [ ]:
def load_dataset(name):
    """Featurize one benchmark, caching the result."""
    spec = DATASETS[name]
    cache = CACHE_DIR / f"{name}.pt"
    if cache.exists():
        payload = torch.load(cache, weights_only=False)
        return payload["graphs"], payload["targets"]

    frame = pd.read_csv(DATA_DIR / spec["file"])
    if spec["targets"] is None:
        drop = set(spec.get("drop", [])) | {spec["smiles"]}
        targets = [c for c in frame.columns if c not in drop]
    else:
        targets = spec["targets"]

    missing = [c for c in [spec["smiles"]] + targets if c not in frame.columns]
    if missing:
        raise KeyError(f"{name}: columns {missing} absent. Present: {list(frame.columns)[:15]}")

    smiles = frame[spec["smiles"]].astype(str).tolist()
    labels = frame[targets].apply(pd.to_numeric, errors="coerce").values

    graphs, skipped = [], 0
    for value, label in zip(tqdm(smiles, desc=f"featurize {name}", leave=False), labels):
        graph = featurize(value, label.tolist())
        if graph is None:
            skipped += 1
        else:
            graphs.append(graph)

    torch.save({"graphs": graphs, "targets": targets}, cache)
    print(f"{name}: {len(graphs):,} molecules, {len(targets)} task(s), {skipped} skipped")
    return graphs, targets


def scaffold_split(smiles, fractions=(0.8, 0.1, 0.1)):
    """Deterministic Bemis-Murcko scaffold split, largest groups into train."""
    from collections import defaultdict

    groups = defaultdict(list)
    for index, value in enumerate(smiles):
        mol = Chem.MolFromSmiles(value)
        key = MurckoScaffold.MurckoScaffoldSmiles(mol=mol) if mol is not None else ""
        groups[key].append(index)

    ordered = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    total = len(smiles)
    train_cut, valid_cut = fractions[0] * total, (fractions[0] + fractions[1]) * total

    train, valid, test = [], [], []
    for group in ordered:
        if len(train) + len(group) <= train_cut:
            train += group
        elif len(train) + len(valid) + len(group) <= valid_cut:
            valid += group
        else:
            test += group
    return train, valid, test


def random_split(n, seed=42, fractions=(0.8, 0.1, 0.1)):
    permutation = np.random.default_rng(seed).permutation(n)
    a, b = int(fractions[0] * n), int((fractions[0] + fractions[1]) * n)
    return permutation[:a].tolist(), permutation[a:b].tolist(), permutation[b:].tolist()


print("Dataset loading ready.")

## Model

In [ ]:
def mlp(a, b, c):
    return nn.Sequential(nn.Linear(a, b), nn.ReLU(), nn.Linear(b, c))


class GraphEncoder(nn.Module):
    """Must match the pretraining architecture exactly, LayerNorm included."""

    def __init__(self, node_dim, edge_dim, hidden_dim=128, output_dim=128,
                 num_layers=3, dropout=0.0):
        super().__init__()
        self.node_encoder = mlp(node_dim, hidden_dim, hidden_dim)
        self.edge_encoder = mlp(edge_dim, hidden_dim, hidden_dim)
        self.convs = nn.ModuleList([
            GCNConv(hidden_dim, output_dim if i == num_layers - 1 else hidden_dim)
            for i in range(num_layers)
        ])
        self.dropout = dropout
        self.pool_norm = nn.LayerNorm(output_dim)
        self.projection = mlp(output_dim, hidden_dim, output_dim)

    def encode(self, data, feature_mean, feature_std, clamp):
        x = torch.cat([data.x_cat.float(), data.x_phys.float()], dim=-1)
        x = torch.clamp((x - feature_mean) / feature_std, -clamp, clamp)
        x = self.node_encoder(x)
        _ = self.edge_encoder(data.edge_attr.float())

        edge_weight = getattr(data, "edge_weight", None)
        for index, conv in enumerate(self.convs):
            x = conv(x, data.edge_index, edge_weight)
            if index < len(self.convs) - 1:
                x = F.relu(x)
                if self.dropout > 0:
                    x = F.dropout(x, p=self.dropout, training=self.training)

        batch = getattr(data, "batch", None)
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return self.pool_norm(global_mean_pool(x, batch))


class DownstreamModel(nn.Module):
    def __init__(self, encoder, num_tasks, stats, dropout=0.2, freeze=False):
        super().__init__()
        self.encoder = encoder
        self.freeze = freeze
        if freeze:
            for parameter in encoder.parameters():
                parameter.requires_grad = False
        self.mean, self.std, self.clamp = stats
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(encoder.convs[-1].out_channels, num_tasks)
        )

    def forward(self, data):
        if self.freeze:
            with torch.no_grad():
                self.encoder.eval()
                features = self.encoder.encode(data, self.mean, self.std, self.clamp)
        else:
            features = self.encoder.encode(data, self.mean, self.std, self.clamp)
        return self.head(features)


def build_encoder(checkpoint_path):
    """Rebuild the pretrained encoder and its feature statistics.

    The projection head is discarded: the pooled representation is the transfer
    target, since the projection is trained to satisfy the contrastive
    objective and discards information it does not need.
    """
    node_dim, edge_dim = 10, NUM_EDGE_FEATURES

    if checkpoint_path is None:
        encoder = GraphEncoder(node_dim, edge_dim)
        for module in encoder.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)
        stats_dir = next((p.parent for p in
                          (WORKSPACE / "runs").glob("*/feature_stats.pt")), None)
        stats_file = stats_dir / "feature_stats.pt" if stats_dir else None
    else:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        config = checkpoint.get("config", {})
        encoder = GraphEncoder(
            node_dim=config.get("node_dim", node_dim),
            edge_dim=config.get("edge_dim", edge_dim),
            hidden_dim=config.get("hidden_dim", 128),
            output_dim=config.get("output_dim", 128),
            num_layers=config.get("num_layers", 3),
            dropout=config.get("dropout", 0.1),
        )
        missing, unexpected = encoder.load_state_dict(
            checkpoint["encoder_state_dict"], strict=False
        )
        if missing:
            print(f"    WARNING missing keys: {missing}")
        stats_file = Path(checkpoint_path).parent / "feature_stats.pt"

    if stats_file is None or not Path(stats_file).exists():
        raise FileNotFoundError("feature_stats.pt not found; it is written during pretraining.")

    stats = torch.load(stats_file, map_location=DEVICE, weights_only=False)
    return encoder, (stats["mean"].to(DEVICE), stats["std"].to(DEVICE), stats["clamp"])


print("Model ready.")

## MetricsPer task, then averaged over tasks, with missing labels masked out. Tox21 hasabout 17% missing entries, so masking is not optional there.

In [ ]:
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)


def classification_metrics(targets, logits):
    """ROC-AUC, PR-AUC, F1, balanced accuracy and MCC, averaged over tasks.

    PR-AUC and MCC are included because several of these benchmarks are
    strongly imbalanced, where ROC-AUC alone overstates performance.
    """
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    scores = {k: [] for k in ["roc_auc", "pr_auc", "f1", "balanced_accuracy", "mcc"]}

    for task in range(targets.shape[1]):
        mask = ~np.isnan(targets[:, task])
        y, p = targets[mask, task], probabilities[mask, task]
        if y.size == 0 or np.unique(y).size < 2:
            continue
        predicted = (p >= 0.5).astype(int)
        scores["roc_auc"].append(roc_auc_score(y, p))
        scores["pr_auc"].append(average_precision_score(y, p))
        scores["f1"].append(f1_score(y, predicted, zero_division=0))
        scores["balanced_accuracy"].append(balanced_accuracy_score(y, predicted))
        scores["mcc"].append(matthews_corrcoef(y, predicted))

    out = {k: float(np.mean(v)) if v else float("nan") for k, v in scores.items()}
    out["tasks_scored"] = len(scores["roc_auc"])
    return out


def regression_metrics(targets, predictions):
    """RMSE, MAE, R-squared and rank/linear correlation, averaged over tasks.

    R-squared is reported because RMSE is not comparable across datasets with
    different target variances: a small RMSE on a low-variance target can still
    mean the model has learned nothing.
    """
    scores = {k: [] for k in ["rmse", "mae", "r2", "pearson", "spearman"]}

    for task in range(targets.shape[1]):
        mask = ~np.isnan(targets[:, task])
        y, p = targets[mask, task], predictions[mask, task]
        if y.size < 3:
            continue
        scores["rmse"].append(float(np.sqrt(mean_squared_error(y, p))))
        scores["mae"].append(float(mean_absolute_error(y, p)))
        scores["r2"].append(float(r2_score(y, p)))
        if np.std(p) > 1e-12:
            scores["pearson"].append(float(pearsonr(y, p)[0]))
            scores["spearman"].append(float(spearmanr(y, p).correlation))

    out = {k: float(np.mean(v)) if v else float("nan") for k, v in scores.items()}
    out["tasks_scored"] = len(scores["rmse"])
    return out


HIGHER_IS_BETTER = {"roc_auc", "pr_auc", "f1", "balanced_accuracy", "mcc",
                    "r2", "pearson", "spearman"}
print("Metrics ready.")

## Training one configuration

In [ ]:
def masked_loss(outputs, targets, task):
    mask = ~torch.isnan(targets)
    if mask.sum() == 0:
        return outputs.sum() * 0.0
    if task == "classification":
        return F.binary_cross_entropy_with_logits(outputs[mask], targets[mask])
    return F.mse_loss(outputs[mask], targets[mask])


def evaluate(model, loader, task, scaler):
    model.eval()
    outputs, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            outputs.append(model(batch).cpu().numpy())
            targets.append(batch.y.view(batch.num_graphs, -1).float().cpu().numpy())
    outputs = np.concatenate(outputs)
    targets = np.concatenate(targets)
    if task == "regression" and scaler is not None:
        outputs = outputs * scaler[1] + scaler[0]
    return (classification_metrics(targets, outputs) if task == "classification"
            else regression_metrics(targets, outputs))


def run_one(name, checkpoint_path, seed):
    spec = DATASETS[name]
    task = spec["task"]
    graphs, targets = load_dataset(name)

    torch.manual_seed(seed); np.random.seed(seed)

    smiles = [g.smiles for g in graphs]
    split = (scaffold_split(smiles) if spec["split"] == "scaffold"
             else random_split(len(graphs), seed=42))
    train_idx, valid_idx, test_idx = split

    # Regression targets are standardised using TRAIN statistics only, and
    # predictions are inverse-transformed before scoring, so reported RMSE and
    # MAE stay in the original units. QM7 in particular has targets in the
    # thousands, which would otherwise dominate the loss.
    scaler = None
    if task == "regression":
        train_y = np.stack([graphs[i].y.view(-1).numpy() for i in train_idx])
        mean, std = np.nanmean(train_y, 0), np.nanstd(train_y, 0)
        std[std < 1e-8] = 1.0
        scaler = (mean, std)
        for graph in graphs:
            graph.y = torch.tensor(
                ((graph.y.view(-1).numpy() - mean) / std)[None, :], dtype=torch.float
            )

    from torch.utils.data import Subset
    loaders = [
        DataLoader(Subset(graphs, idx), batch_size=BATCH_SIZE,
                   shuffle=(k == 0), drop_last=False)
        for k, idx in enumerate(split)
    ]

    encoder, stats = build_encoder(checkpoint_path)
    model = DownstreamModel(encoder, len(targets), stats, HEAD_DROPOUT,
                            freeze=(PROTOCOL == "linear_probe")).to(DEVICE)

    groups = [{"params": model.head.parameters(), "lr": HEAD_LR}]
    if PROTOCOL != "linear_probe":
        groups.append({"params": model.encoder.parameters(), "lr": ENCODER_LR})
    optimizer = torch.optim.Adam(groups, weight_decay=WEIGHT_DECAY)

    primary = spec["primary"]
    higher = primary in HIGHER_IS_BETTER
    best_valid = -np.inf if higher else np.inf
    best_test, stalled = {}, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        if model.freeze:
            model.encoder.eval()
        for batch in loaders[0]:
            batch = batch.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = masked_loss(model(batch), batch.y.view(batch.num_graphs, -1).float(), task)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        valid = evaluate(model, loaders[1], task, scaler)[primary]
        improved = (valid > best_valid) if higher else (valid < best_valid)
        if improved and not np.isnan(valid):
            best_valid, stalled = valid, 0
            best_test = evaluate(model, loaders[2], task, scaler)
        else:
            stalled += 1
            if stalled >= PATIENCE:
                break

    best_test[f"valid_{primary}"] = float(best_valid)
    best_test["epochs_run"] = epoch
    return best_test


print("Runner ready.")

## Run

In [ ]:
results = {}
results_path = RESULTS_DIR / f"results_{PROTOCOL}.json"
if results_path.exists():
    with open(results_path, encoding="utf-8") as handle:
        results = json.load(handle)

for name in ACTIVE_DATASETS:
    if not (DATA_DIR / DATASETS[name]["file"]).exists():
        print(f"SKIP {name}: file missing")
        continue

    results.setdefault(name, {})
    for label, checkpoint in CHECKPOINTS.items():
        if checkpoint is not None and not Path(checkpoint).exists():
            print(f"SKIP {label} on {name}: checkpoint missing")
            continue
        if label in results[name]:
            continue

        started = time.perf_counter()
        runs = []
        for seed in SEEDS:
            runs.append(run_one(name, checkpoint, seed))

        aggregated = {}
        for key in runs[0]:
            values = np.array([r.get(key, np.nan) for r in runs], dtype=float)
            if np.all(np.isnan(values)):
                continue
            aggregated[key] = {"mean": float(np.nanmean(values)),
                               "std": float(np.nanstd(values))}
        results[name][label] = {"runs": runs, "summary": aggregated}

        primary = DATASETS[name]["primary"]
        cell = aggregated.get(primary, {})
        print(f"{name:<14} {label:<16} {primary} "
              f"{cell.get('mean', float('nan')):.4f} +/- {cell.get('std', float('nan')):.4f} "
              f"({(time.perf_counter() - started)/60:.1f} min)")

        with open(results_path, "w", encoding="utf-8") as handle:
            json.dump(results, handle, indent=2)

print(f"\nWritten to {results_path}")

## Results tables

In [ ]:
CLASSIFICATION_METRICS = ["roc_auc", "pr_auc", "f1", "balanced_accuracy", "mcc"]
REGRESSION_METRICS = ["rmse", "mae", "r2", "pearson", "spearman"]


def print_table(task):
    names = [n for n in ACTIVE_DATASETS
             if DATASETS[n]["task"] == task and n in results and results[n]]
    if not names:
        return
    metrics = CLASSIFICATION_METRICS if task == "classification" else REGRESSION_METRICS

    for metric in metrics:
        print(f"\n{metric.upper()}  ({'higher' if metric in HIGHER_IS_BETTER else 'lower'} is better)")
        header = f"{'model':<18}" + "".join(f"{n:>16}" for n in names)
        print(header); print("-" * len(header))
        for label in CHECKPOINTS:
            row = f"{label:<18}"
            for name in names:
                cell = results[name].get(label, {}).get("summary", {}).get(metric)
                row += (f"{cell['mean']:>10.3f}+-{cell['std']:<4.3f}" if cell else f"{'-':>16}")
            print(row)


print("=" * 70); print("CLASSIFICATION"); print("=" * 70)
print_table("classification")
print()
print("=" * 70); print("REGRESSION"); print("=" * 70)
print_table("regression")

In [ ]:
import csv

csv_path = RESULTS_DIR / f"results_{PROTOCOL}.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["dataset", "task", "n_tasks", "model", "metric", "mean", "std"])
    for name in results:
        task = DATASETS[name]["task"]
        metrics = CLASSIFICATION_METRICS if task == "classification" else REGRESSION_METRICS
        for label, payload in results[name].items():
            for metric in metrics:
                cell = payload["summary"].get(metric)
                if cell:
                    writer.writerow([name, task, payload["summary"].get("tasks_scored", {}).get("mean", ""),
                                     label, metric, f"{cell['mean']:.4f}", f"{cell['std']:.4f}"])
print(f"Wrote {csv_path}")

## Reading the results**Compare against the "No pretraining" row first.** If a pretrained encoderdoes not beat a randomly initialised one, the pretraining contributed nothingon that benchmark, whatever the absolute number looks like.**On the imbalanced benchmarks, read PR-AUC before ROC-AUC.** HIV is about3.5% actives and ClinTox about 7% toxic. A model that ranks the few positivesmoderately well can post a respectable ROC-AUC while being of little practicaluse; PR-AUC and MCC expose that.**On regression, read R-squared next to RMSE.** An R-squared at or below zeromeans the predictions are no better than the training mean, which an RMSE valuealone will not reveal.**Quote the effective rank from the embedding analysis alongside thesenumbers.** A collapsed encoder still produces downstream metrics, and withoutthe rank a reader cannot tell whether a weak result came from a degeneraterepresentation or from a genuinely hard task.QM7 and QM8 use MAE and a random split, following the convention in theliterature; the others use RMSE or ROC-AUC with a scaffold split.